# Temporal Dynamics Analysis

Analyzing time-varying neural representations:
- Dynamic Time Warping (DTW) for sequence alignment
- Inter-Subject Synchronization (ISC)
- Time-Resolved Canonical Correlation Analysis (TR-CCA)
- Temporal Receptive Fields (TRF)
- Phase precession analysis

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from neuros_mechint.alignment import (
    DynamicTimeWarping,
    InterSubjectSynchronization,
    TimeResolvedCCA,
    TemporalReceptiveField,
    PhasePrecession
)
from neuros_mechint.visualization import TemporalDynamicsVisualizer

%matplotlib inline

## 1. Dynamic Time Warping

Align two temporal sequences with variable timing.

In [ ]:
# Create two similar sequences with time warping
t = np.linspace(0, 4*np.pi, 100)
seq1 = np.sin(t) + 0.2 * np.random.randn(100)

# Seq2: warped version of seq1
t_warped = t + 0.5 * np.sin(t * 0.5)  # Non-linear time warp
seq2 = np.sin(t_warped) + 0.2 * np.random.randn(100)

# Perform DTW
dtw = DynamicTimeWarping()
distance, path = dtw.compute(seq1.reshape(-1, 1), seq2.reshape(-1, 1))

print(f"DTW distance: {distance:.4f}")
print(f"Alignment path length: {len(path)}")

# Visualize
plt.figure(figsize=(14, 8))

# Original sequences
plt.subplot(3, 1, 1)
plt.plot(t, seq1, 'b-', label='Sequence 1', linewidth=2)
plt.plot(t, seq2, 'r-', label='Sequence 2', linewidth=2)
plt.ylabel('Amplitude')
plt.title('Original Sequences (with time warp)')
plt.legend()
plt.grid(True, alpha=0.3)

# Cost matrix with alignment path
plt.subplot(3, 1, 2)
plt.imshow(dtw.cost_matrix, aspect='auto', cmap='viridis', origin='lower')
plt.colorbar(label='Cumulative Cost')
path_array = np.array(path)
plt.plot(path_array[:, 1], path_array[:, 0], 'r-', linewidth=2, label='Optimal Path')
plt.xlabel('Sequence 2 Index')
plt.ylabel('Sequence 1 Index')
plt.title('DTW Cost Matrix with Alignment Path')
plt.legend()

# Aligned sequences
plt.subplot(3, 1, 3)
aligned_time = np.arange(len(path))
aligned_seq1 = [seq1[i] for i, j in path]
aligned_seq2 = [seq2[j] for i, j in path]
plt.plot(aligned_time, aligned_seq1, 'b-', label='Sequence 1 (aligned)', linewidth=2, alpha=0.7)
plt.plot(aligned_time, aligned_seq2, 'r-', label='Sequence 2 (aligned)', linewidth=2, alpha=0.7)
plt.xlabel('Aligned Time')
plt.ylabel('Amplitude')
plt.title('DTW-Aligned Sequences')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compare correlations
orig_corr = np.corrcoef(seq1, seq2)[0, 1]
aligned_corr = np.corrcoef(aligned_seq1, aligned_seq2)[0, 1]
print(f"\nCorrelation before alignment: {orig_corr:.4f}")
print(f"Correlation after alignment:  {aligned_corr:.4f}")
print(f"Improvement: {(aligned_corr - orig_corr):.4f}")

## 2. Inter-Subject Synchronization

Measure shared responses across subjects/samples.

In [ ]:
# Simulate multi-subject neural data
n_subjects = 10
n_timepoints = 200
n_features = 50

# Create shared signal + individual noise
shared_signal = np.random.randn(n_timepoints, n_features)
subjects_data = []

for i in range(n_subjects):
    # Mix shared and idiosyncratic components
    shared_weight = 0.7  # 70% shared
    individual_signal = np.random.randn(n_timepoints, n_features)
    subject_data = shared_weight * shared_signal + (1 - shared_weight) * individual_signal
    subjects_data.append(subject_data)

subjects_data = np.array(subjects_data)  # (n_subjects, n_timepoints, n_features)

# Compute ISC
isc = InterSubjectSynchronization()
isc_values = isc.compute(subjects_data)

print(f"ISC results:")
print(f"  Mean ISC: {isc_values.mean():.4f}")
print(f"  Max ISC: {isc_values.max():.4f}")
print(f"  Min ISC: {isc_values.min():.4f}")

# Visualize
plt.figure(figsize=(14, 5))

# ISC distribution
plt.subplot(1, 3, 1)
plt.hist(isc_values, bins=20, edgecolor='black', alpha=0.7)
plt.axvline(isc_values.mean(), color='r', linestyle='--', linewidth=2, label=f'Mean = {isc_values.mean():.3f}')
plt.xlabel('ISC Value')
plt.ylabel('Count')
plt.title('ISC Distribution Across Features')
plt.legend()
plt.grid(True, alpha=0.3)

# Time course of high ISC feature
high_isc_feature = np.argmax(isc_values)
plt.subplot(1, 3, 2)
for i in range(min(5, n_subjects)):
    plt.plot(subjects_data[i, :, high_isc_feature], alpha=0.7, label=f'S{i+1}')
plt.xlabel('Time')
plt.ylabel('Activity')
plt.title(f'Highest ISC Feature (ISC={isc_values[high_isc_feature]:.3f})')
plt.legend()
plt.grid(True, alpha=0.3)

# Time course of low ISC feature
low_isc_feature = np.argmin(isc_values)
plt.subplot(1, 3, 3)
for i in range(min(5, n_subjects)):
    plt.plot(subjects_data[i, :, low_isc_feature], alpha=0.7, label=f'S{i+1}')
plt.xlabel('Time')
plt.ylabel('Activity')
plt.title(f'Lowest ISC Feature (ISC={isc_values[low_isc_feature]:.3f})')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Time-Resolved CCA

Track temporal evolution of cross-correlations.

In [ ]:
# Two time-varying neural populations
n_timepoints = 500
n_dims_X = 30
n_dims_Y = 30

# Create time-varying correlation
t = np.linspace(0, 4*np.pi, n_timepoints)
coupling_strength = 0.5 + 0.4 * np.sin(t)  # Oscillating coupling

X = np.random.randn(n_timepoints, n_dims_X)
Y = np.random.randn(n_timepoints, n_dims_Y)

# Add time-varying coupling
for i in range(n_timepoints):
    Y[i, :5] = coupling_strength[i] * X[i, :5] + (1 - coupling_strength[i]) * Y[i, :5]

# Compute time-resolved CCA
trcca = TimeResolvedCCA(window_size=50, step_size=10)
correlations = trcca.compute(X, Y)

time_points = np.arange(len(correlations)) * trcca.step_size + trcca.window_size // 2

# Visualize
plt.figure(figsize=(12, 6))

plt.subplot(2, 1, 1)
plt.plot(t, coupling_strength, 'k-', linewidth=2, label='True Coupling')
plt.ylabel('Coupling Strength')
plt.title('Time-Varying Neural Coupling')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 1, 2)
plt.plot(time_points, correlations, 'b-', linewidth=2, marker='o', markersize=4, label='TR-CCA')
plt.xlabel('Time')
plt.ylabel('Canonical Correlation')
plt.title('Time-Resolved Canonical Correlation')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Correlation between true and estimated
# Interpolate to match time points
true_coupling_interp = np.interp(time_points, np.arange(n_timepoints), coupling_strength)
tracking_corr = np.corrcoef(true_coupling_interp, correlations)[0, 1]

print(f"TR-CCA tracking accuracy: r = {tracking_corr:.4f}")

## 4. Temporal Receptive Fields

Estimate how past stimulus drives neural response.

In [ ]:
# Simulate stimulus and response
n_timepoints = 1000
stimulus = np.random.randn(n_timepoints)

# Create TRF (decaying response over 50 ms)
true_trf = np.exp(-np.arange(50) / 10)
true_trf /= true_trf.sum()

# Generate response via convolution
response = np.convolve(stimulus, true_trf, mode='same')
response += np.random.randn(n_timepoints) * 0.2  # Add noise

# Estimate TRF
trf_estimator = TemporalReceptiveField(window_size=50)
estimated_trf = trf_estimator.estimate(stimulus, response)

# Visualize
plt.figure(figsize=(14, 5))

plt.subplot(1, 3, 1)
plt.plot(stimulus[:200], 'b-', linewidth=1, label='Stimulus')
plt.plot(response[:200], 'r-', linewidth=1, label='Response')
plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.title('Stimulus and Response')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(true_trf, 'k-', linewidth=2, label='True TRF')
plt.plot(estimated_trf, 'r--', linewidth=2, label='Estimated TRF')
plt.xlabel('Lag (timesteps)')
plt.ylabel('Weight')
plt.title('Temporal Receptive Field')
plt.legend()
plt.grid(True, alpha=0.3)

# Prediction accuracy
predicted_response = np.convolve(stimulus, estimated_trf, mode='same')
plt.subplot(1, 3, 3)
plt.scatter(response, predicted_response, alpha=0.3, s=10)
plt.plot([response.min(), response.max()], [response.min(), response.max()], 
         'r--', linewidth=2, label='Perfect prediction')
plt.xlabel('Actual Response')
plt.ylabel('Predicted Response')
pred_corr = np.corrcoef(response, predicted_response)[0, 1]
plt.title(f'Prediction Accuracy (r = {pred_corr:.4f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')

plt.tight_layout()
plt.show()

print(f"\nTRF Estimation Results:")
print(f"  Peak latency: {np.argmax(estimated_trf)} timesteps")
print(f"  TRF correlation with true: {np.corrcoef(true_trf, estimated_trf)[0,1]:.4f}")
print(f"  Response prediction: r = {pred_corr:.4f}")

## Conclusion

This notebook demonstrated:
1. **DTW**: Aligning sequences with variable timing
2. **ISC**: Measuring shared responses across subjects
3. **TR-CCA**: Tracking time-varying correlations
4. **TRF**: Estimating stimulus-response relationships

These methods reveal how neural representations evolve over time, essential for understanding temporal processing in both brains and foundation models.